# Q13 — Scheduling interference with hardware-informed memory decay

**Upload this notebook to Jupyter on your server and select Run All. Only this file is required.**
It embeds the code, source-derived parameters, and mathematical reference. It uses the same validated
P1 NetSquid/Qoala environment. It does not install packages or alter the Qoala checkout.

**This study can run while Q12 is running.** It uses a separate `q13-...` results directory, one worker by
default, and one numerical-library thread per worker. It neither reads Q12 results nor changes that study.
Keep the server allocation alive. If Q12 occupies six of eight allocated cores and no other jobs are
using the remainder, `WORKERS=1` or `2` is suitable. Increase workers only when more cores are available.

The default `MODE="full"` automatically runs **NetSquid baseline → integration pilot → robustness evaluation → export**.
It stops on any validation failure. To run a shorter check, use `MODE="baseline"` or `MODE="pilot"`.
Later, set `MODE="full"` and keep `RESUME_RUN_DIR="auto"`; completed batches are verified and reused.

| Stage | Work items | Purpose |
|---|---:|---|
| NetSquid baseline | 336 numerical cases | Two EPR matrices, two parameterizations, six circuits, seven ages, whole/split noise application; both client branches required per case |
| Integration pilot | 1,320 executions | 1,176 noise-free controls and 144 noisy checks; actual CPU chronology, equal work, correction branches and quantum states |
| Robustness evaluation | 73,728 executions | Two workloads × two memory profiles × three timing uncertainties × three arms × 32 independent replications × 64 trials |

Each numerical baseline case uses a bounded seeded search to exercise both client-measurement branches.
Those attempts are correctness checks, not data used to estimate an attack effect.

## What is tested

The baseline checks the audited QNodeOS delegated-computation circuit against independent NumPy density matrices.
It includes the normalized rounded EPR matrix from the final supplement and the precise matrix regenerated from
the deposited model. The original deposited numerical sweep used one circuit setting; checking all six is an extension.

The separate Qoala experiment asks whether **fixed release timing increases simulated state infidelity more than
equal-work random timing** when memory decays according to either source parameterization:

| Profile | Coherence scale | Stretched exponent |
|---|---:|---:|
| Final Supplementary Information | 13 ms | 1.67 |
| Deposited simulation code | 13.2 ms | 1.63 |

For either profile the single-qubit channel is
\[
\rho(t)=D(t)\rho(0)+(1-D(t))I/2,\qquad D(t)=e^{-(t/T)^n}.
\]
Splitting storage across events preserves total age, using contraction `D(end)/D(start)`.
In these ideal P1 programs, isotropic noise commutes with local unitary gates and remote operations.
The full-age channel is therefore applied once immediately before verification, and NetSquid then performs the
actual measurement. Native T1/T2 noise is disabled, avoiding double counting. This equivalence does not cover
arbitrary noisy gates, nonunital channels or local adaptive measurement programs.

**Frozen schedules:** the competing workload is four ordinary 30-instruction CPU jobs, totaling 1.2 ms of service.
The fixed offsets are 1.25 ms for the plus-state workload and 0.5 ms for teleportation. They are existing Q12
integration-control constants, not learned from Q12 results and not claimed optimal. Random timing is uniform over
0–4 ms from the same declared epoch. All samples and phases are fixed before evaluation; no phase search occurs here.

The primary family contains four fixed-versus-random contrasts at **±0.5 ms request uncertainty**.
Exact request timing and ±1 ms uncertainty are exploratory checks. No-competitor controls are retained throughout.
Every slow or unfinished execution is retained. There is **no 8.95 ms postselection cutoff**.

## Interpretation boundary

The Qoala study transfers the source **memory-decay envelope**, not the complete hardware platform. It retains
ideal initial states/EPR pairs and local gates, FCFS, 10 μs per host instruction, 1 μs per QNOS instruction,
1 μs ideal EPR generation and 0.5–1.5 ms one-way classical messages. These timing choices are P1 assumptions.
The source's 10 μs QNPU timestamp tick is not a measurement of host-instruction duration. Storage here ends at the
first verification gate; the audited hardware trace ends at measurement-command issuance.

Shared placement and a coarse request epoch remain explicit assumptions. Q13 tests physical-model robustness;
it does not establish a deployed timing API, learned-strategy transfer, or an attack on hardware. Q12 addresses
the separate self-calibration question.

**Validation before delivery:** nine offline checks passed. NetSquid is unavailable in the development environment;
the new simulator code has **not** been runtime-executed there. The mandatory server baseline and pilot provide
that integration check before the large evaluation. A passing gate is not evidence of a positive attack effect.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.home() / "P1"
MODE = "full"
WORKERS = 1

# "auto": resume the latest run created by this exact code bundle, or start one.
# None: explicitly start a NEW run. A full path: resume that particular run.
RESUME_RUN_DIR = "auto"

# Usually leave these as None. The next cells locate your existing environment.
PYTHON = None   # Example: PROJECT_DIR / ".venv-p1" / "bin" / "python"
REPO = None     # Example: PROJECT_DIR / "qoala-sim"

# The review ZIP includes outcomes, analysis, code, metadata and diagnostic logs.
# Raw traces stay on the server. Set True later if a full trace archive is needed.
EXPORT_RAW = False


In [ ]:
import base64, fcntl, gzip, hashlib, json, os, sys

SOURCE_BUNDLE_SHA256 = 'aa79f9d3855ba1056cefe5c66c51a729ca9ee19bda2610bf2f4acd8e8a7a29c9'
_EMBEDDED = '...'
_EXPECTED_FILES = ['q13.py', 'q13_design.py', 'q13_engine.py', 'q13_baseline.py', 'q13_analysis.py', 'q13_tests.py', 'p1_engine.py', 'q6_reference.py', 'time_checks.py', 'physics_reference.py', 'hardware_models.json', 'source_provenance.json']

def _sha_bytes(value):
    return hashlib.sha256(value).hexdigest()

def install_embedded_code(project):
    data = gzip.decompress(base64.b64decode(_EMBEDDED))
    if _sha_bytes(data) != SOURCE_BUNDLE_SHA256:
        raise RuntimeError("Embedded source bundle checksum mismatch")
    files = json.loads(data)
    if set(files) != set(_EXPECTED_FILES):
        raise RuntimeError("Unexpected embedded source files")
    base = Path(project).expanduser().resolve() / "experiment_code"
    base.mkdir(parents=True, exist_ok=True)
    folder = base / ("q13-" + SOURCE_BUNDLE_SHA256[:16])
    with (base / (folder.name + ".install.lock")).open("a+") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        folder.mkdir(exist_ok=True)
        for name, content in files.items():
            if Path(name).name != name:
                raise RuntimeError("Invalid embedded filename")
            expected = content.encode()
            destination = folder / name
            if destination.exists():
                if destination.read_bytes() != expected:
                    raise RuntimeError("Installed source differs: " + str(destination) +
                                       ". Preserve it; do not overwrite or mix versions.")
            else:
                temporary = destination.with_name(name + ".tmp-" + str(os.getpid()))
                temporary.write_bytes(expected)
                os.replace(temporary, destination)
            if _sha_bytes(destination.read_bytes()) != _sha_bytes(expected):
                raise RuntimeError("Source verification failed: " + name)
    return folder

PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
CODE_DIR = install_embedded_code(PROJECT_DIR)
print("Embedded code installed and verified:", CODE_DIR, flush=True)
print("Source bundle hash:", SOURCE_BUNDLE_SHA256, flush=True)


In [ ]:
import shutil, subprocess

if MODE not in ("baseline", "pilot", "full"):
    raise ValueError('MODE must be "baseline", "pilot" or "full"')
if not isinstance(WORKERS, int) or isinstance(WORKERS, bool) or WORKERS < 1:
    raise ValueError("WORKERS must be a positive integer")
available_cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if available_cpus and WORKERS > available_cpus:
    raise ValueError(f"WORKERS={WORKERS}, but this process can use only {available_cpus} CPUs")

RUN_ENV = os.environ.copy()
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    RUN_ENV[variable] = "1"
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV.pop("PYTHONOPTIMIZE", None)

repo_candidates = ([Path(REPO).expanduser()] if REPO is not None else
                   [PROJECT_DIR / name for name in ("qoala-sim", "p1-qoala", "qoala")] +
                   [Path.home() / "p1-qoala", Path.home() / "qoala-sim"])
repos = []
for candidate in repo_candidates:
    if not candidate.is_dir():
        continue
    result = subprocess.run(["git", "-C", str(candidate), "rev-parse", "HEAD"],
                            capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip() == "31fa2c9699578d227736557f35cd01a134c4bbde":
        repos.append(candidate.resolve())
if not repos:
    raise RuntimeError("Could not locate the existing pinned Qoala checkout. Set REPO in the configuration "
                       "cell to its directory, then rerun. Checked: " + ", ".join(map(str, repo_candidates)))
REPO = repos[0]

if PYTHON is not None:
    located = shutil.which(str(PYTHON))
    python_candidates = [located or str(Path(PYTHON).expanduser().absolute())]
else:
    # Do not resolve interpreter symlinks: doing so can bypass a virtual environment.
    python_candidates = [str(PROJECT_DIR / ".venv-p1/bin/python"),
                         str(PROJECT_DIR / ".venv/bin/python"), sys.executable]
python_candidates = list(dict.fromkeys(python_candidates))
preflight = "import json,sys; import q6_reference; info=q6_reference.check_repo(sys.argv[1]); " \
            "import scipy; info['scipy']=scipy.__version__; print(json.dumps(info,indent=2))"
failures = []
chosen = None
for candidate in python_candidates:
    if not Path(candidate).is_file():
        continue
    result = subprocess.run([candidate, "-c", preflight, str(REPO)], cwd=CODE_DIR,
                            env=RUN_ENV, capture_output=True, text=True)
    if result.returncode == 0:
        chosen = candidate
        print("Using Python:", candidate, flush=True)
        print("Using Qoala:", REPO, flush=True)
        print(result.stdout, flush=True)
        break
    failures.append(candidate + "\n" + result.stdout + result.stderr)
if chosen is None:
    print("\n".join(failures), flush=True)
    raise RuntimeError("No validated environment found. Set PYTHON to the same interpreter used for "
                       "Q11/Q9S. Required: Python 3.10, NetSquid 1.1.8, pydynaa 1.0.2, Qoala 2.1.0, "
                       "NumPy 1.26.4, netqasm 2.3.0 and SciPy. This notebook has not changed your environment.")
PYTHON = chosen

# Capture and print so output appears reliably in Jupyter as well as a terminal.
tests = subprocess.run([PYTHON, "-u", str(CODE_DIR / "q13_tests.py")],
                       cwd=CODE_DIR, env=RUN_ENV, capture_output=True, text=True)
print(tests.stdout, end="", flush=True)
if tests.stderr:
    print(tests.stderr, end="", flush=True)
if tests.returncode:
    raise RuntimeError("Local design checks failed; do not start the experiment")
print(f"Preflight passed. Mode: {MODE} | workers: {WORKERS}. The next cell runs the NetSquid baseline checks and integration pilot.", flush=True)


## Run or resume

The next cell prints **RUN DIRECTORY** immediately and streams progress. It remains busy while simulations run.
The numerical baseline must pass first; the integration pilot then prints an approximate full-evaluation runtime.
The full default continues automatically. Check the printed stages, not merely the earlier offline-test output.

If the session ends, reopen **this same notebook** and Run All with `RESUME_RUN_DIR="auto"`.
An explicit printed `q13-...` path also works. An incomplete batch is rerun; completed batches are hash-verified
and reused. You can change `WORKERS` on resume, keeping the code and protocol unchanged.
Run locking rejects a second coordinator for the same folder. Q12 has its own folder and coordinator.


In [ ]:
from datetime import datetime, timezone
import signal, time

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
POINTER = RESULTS_DIR / (".q13-latest-" + SOURCE_BUNDLE_SHA256[:16] + ".json")

def choose_run_directory(resume):
    if resume == "auto" and POINTER.exists():
        previous = json.loads(POINTER.read_text())
        if previous["source_bundle_sha256"] != SOURCE_BUNDLE_SHA256:
            raise RuntimeError("Resume pointer belongs to a different source bundle")
        root = Path(previous["run_dir"]).expanduser().resolve()
        if not root.is_dir():
            raise RuntimeError("The saved run folder is missing: " + str(root) +
                               ". Restore it, or set RESUME_RUN_DIR=None to explicitly start anew.")
        return root
    if resume is None or resume == "auto":
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        return RESULTS_DIR / ("q13-" + stamp)
    root = Path(resume).expanduser().resolve()
    if not root.is_dir():
        raise RuntimeError("Resume folder does not exist: " + str(root))
    if not root.name.startswith("q13-"):
        raise RuntimeError("Expected a Q13 run folder; do not reuse a Q12/Q11/Q9S folder")
    return root

RUN_DIR = choose_run_directory(RESUME_RUN_DIR)
if RUN_DIR == REPO or REPO in RUN_DIR.parents or RUN_DIR == CODE_DIR or CODE_DIR in RUN_DIR.parents:
    raise RuntimeError("Results must be outside the source and Qoala checkout folders")
RUN_DIR.mkdir(parents=True, exist_ok=True)
pointer_temp = POINTER.with_name(POINTER.name + ".tmp-" + str(os.getpid()))
pointer_temp.write_text(json.dumps(dict(run_dir=str(RUN_DIR), source_bundle_sha256=SOURCE_BUNDLE_SHA256), indent=2))
os.replace(pointer_temp, POINTER)

print("RUN DIRECTORY:", RUN_DIR, flush=True)
print("For an explicit resume, set RESUME_RUN_DIR =", repr(str(RUN_DIR)), flush=True)
print('Keep MODE="full" to continue all stages, or choose baseline/pilot for a shorter check. The default "auto" remembers this path.', flush=True)
launcher_log = RUN_DIR / ("launcher-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".log")
command = [PYTHON, "-u", str(CODE_DIR / "q13.py"), "run", "--repo", str(REPO),
           "--run-dir", str(RUN_DIR), "--mode", MODE, "--workers", str(WORKERS)]

with launcher_log.open("w") as handle:
    process = subprocess.Popen(command, cwd=CODE_DIR, env=RUN_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    print("Coordinator PID:", process.pid, "| log:", launcher_log, flush=True)
    try:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
        exit_code = process.wait()
    except BaseException:
        # Stop the coordinator and its worker subprocesses together on interruption.
        for sig in (signal.SIGINT, signal.SIGTERM, signal.SIGKILL):
            if process.poll() is not None:
                break
            try:
                os.killpg(process.pid, sig)
            except ProcessLookupError:
                break
            try:
                process.wait(timeout=10)
                break
            except subprocess.TimeoutExpired:
                pass
        print("Stopped. Completed checkpoints remain reusable in", RUN_DIR, flush=True)
        raise
    finally:
        process.stdout.close()
if exit_code:
    raise RuntimeError("Q13 stopped with exit code " + str(exit_code) +
                       ". Run the export cell below manually and share the review ZIP, including logs.")
print("Requested stages finished. The next cell displays available results and exports them.", flush=True)


## Results and export

Run the next cell after completion, or **manually after an error**. It prints status even if analysis is not yet available.
The default review ZIP includes compact outcomes, source, manifests, gate results and diagnostic logs. Large raw traces
remain on the server. Set `EXPORT_RAW=True` to export them as well; keep the original folder for resume and full audit.

The primary endpoint is **expected infidelity at the first verification gate**, obtained from the simulated density
matrix. Positive fixed-minus-random values mean more fidelity loss under the fixed schedule. This is simulator ground
truth, not a hardware tomography estimate. Sampled verification errors and unfinished jobs are reported separately.

Intervals use 20,000 paired bootstrap resamples of the **32 independent replication means**, with a four-comparison
Bonferroni adjustment for the primary family (approximate coverage). They describe random timing/input variation
conditional on the frozen model, not uncertainty in physical parameters. Secondary intervals are exploratory.
An exact sign test describes replication direction separately from the mean-effect interval. The 0.5 percentage-point
reporting reference is disclosed and is not a universal practical-success threshold.

Any unfinished compared execution suppresses primary fidelity-effect inference for that contrast. Per-offered loss
bounds and all outcome counts remain available. No evaluation records are discarded or used to choose a better phase.


In [ ]:
import html, zipfile
from IPython.display import display, HTML, FileLink

if "RUN_DIR" not in globals():
    saved_pointer = PROJECT_DIR / "results" / (".q13-latest-" + SOURCE_BUNDLE_SHA256[:16] + ".json")
    if RESUME_RUN_DIR not in (None, "auto"):
        RUN_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    elif saved_pointer.exists():
        RUN_DIR = Path(json.loads(saved_pointer.read_text())["run_dir"])
    else:
        raise RuntimeError("No Q13 run folder is available yet; run the preceding cells first")

print("RUN DIRECTORY:", RUN_DIR, flush=True)
for label, marker in [("NetSquid baseline complete", "BASELINE_COMPLETE.json"),
                      ("Integration pilot complete", "PILOT_COMPLETE.json"),
                      ("Entire study execution complete", "COMPLETE.json")]:
    print(label + ":", (RUN_DIR / marker).exists(), flush=True)

if (RUN_DIR / "analysis.json").exists():
    report = json.loads((RUN_DIR / "analysis.json").read_text())
    headings = ["Workload", "Physical profile", "Fixed − random infidelity (pp)",
                "Adjusted interval (pp)", "All targets finished", "Interval above zero"]
    table_rows = []
    for row in report["primary_contrasts"]:
        interval = row["approximate_adjusted_interval_pp"]
        effect = row["mean_difference_pp"]
        table_rows.append([row["workload"], row["profile"], "Unavailable" if effect is None else f"{effect:.4f}",
                           "Unavailable" if interval is None else f"[{interval[0]:.4f}, {interval[1]:.4f}]",
                           row["all_compared_targets_completed"], row["interval_above_zero"]])
    markup = '<table><thead><tr>' + ''.join('<th>' + html.escape(x) + '</th>' for x in headings) + '</tr></thead><tbody>'
    for row in table_rows:
        markup += '<tr>' + ''.join('<td>' + html.escape(str(x)) + '</td>' for x in row) + '</tr>'
    display(HTML(markup + '</tbody></table>'))
    print("All four primary intervals above zero:", report["all_four_primary_intervals_above_zero"], flush=True)
    print("Full effect sizes, sampled errors, unfinished counts, and secondary conditions are in analysis.json.", flush=True)
    print("This is model-conditioned robustness. It is not a hardware attack or a validation of timing access.", flush=True)
else:
    print("Analysis is not available yet. Exporting available checkpoints and diagnostics.", flush=True)

# Export is read-only with respect to the scientific run folder.
print("Collecting export file list...", flush=True)
all_paths = sorted(p for p in RUN_DIR.rglob("*") if p.is_file() and p.name != "RUNNING.lock")
inventory = []
paths = []
for path in all_paths:
    is_raw = path.name.endswith(".raw.jsonl.gz")
    included = bool(EXPORT_RAW or not is_raw)
    inventory.append(dict(path=str(path.relative_to(RUN_DIR)), bytes=path.stat().st_size,
                          included=included, raw_trace=is_raw))
    if included:
        paths.append(path)
byte_count = sum(item["bytes"] for item in inventory if item["included"])
raw_excluded = sum(item["raw_trace"] and not item["included"] for item in inventory)
flavor = "full" if EXPORT_RAW else "review"
archive_name = "P1_Qoala_Q13_" + RUN_DIR.name + "_" + flavor + ".zip"
ARCHIVE = PROJECT_DIR / archive_name
print(f"Exporting {len(paths):,} files ({byte_count / 1024**2:.1f} MiB before ZIP). "
      f"Raw trace files retained on server and omitted here: {raw_excluded:,}", flush=True)
temporary_archive = ARCHIVE.with_name(ARCHIVE.name + ".partial-" + str(os.getpid()))
started_export = time.monotonic()
last_export_print = started_export
with zipfile.ZipFile(temporary_archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:
    for index, path in enumerate(paths, 1):
        compression = zipfile.ZIP_STORED if path.suffix == ".gz" else zipfile.ZIP_DEFLATED
        archive.write(path, arcname=str(Path(RUN_DIR.name) / path.relative_to(RUN_DIR)), compress_type=compression)
        if index == 1 or index % 100 == 0 or time.monotonic() - last_export_print >= 10 or index == len(paths):
            print(f"Archived {index:,}/{len(paths):,}", flush=True)
            last_export_print = time.monotonic()
    archive.writestr("EXPORT_MANIFEST.json", json.dumps(dict(
        run_dir=str(RUN_DIR), archive_kind=flavor, source_bundle_sha256=SOURCE_BUNDLE_SHA256,
        raw_trace_files_excluded=raw_excluded,
        note="Review archives cannot independently verify omitted raw traces. Keep original raw files for audit.",
        files=inventory), indent=2))
os.replace(temporary_archive, ARCHIVE)
print(f"Results ZIP: {ARCHIVE} ({ARCHIVE.stat().st_size / 1024**2:.1f} MiB)", flush=True)
download = Path.cwd() / archive_name
if download.resolve() != ARCHIVE.resolve():
    print("Copying archive into the notebook working directory for download...", flush=True)
    shutil.copy2(ARCHIVE, download)
display(FileLink(archive_name))
print("Share the review ZIP first. Keep the original run directory for resume and full trace audit.", flush=True)


## What to share and what remains

- Share the **review ZIP** from the last cell, including negative or inconclusive outcomes. Keep all raw files.
- `MODE="full"` and `RESUME_RUN_DIR="auto"` verify and resume this exact experiment. A completed run does not repeat.
- To deliberately start a separate new run, set `RESUME_RUN_DIR=None`. Never delete metadata to bypass a mismatch.
- This notebook needs no Q6/Q8/Q12 result-folder paths and no copy of the 155 MB published dataset. The two matrices
  and source provenance are embedded. It does need your existing pinned Qoala checkout and validated environment.
- A positive result would support robustness to these memory envelopes under the stated timing assumptions. Hardware
  access, admission rights, epoch availability, nonideal EPR/gate transfer and complete system calibration remain separate.

## Verified source provenance

1. Delle Donne et al., *An operating system for executing applications on quantum network nodes*, Nature 639,
   321–328 (2025). [Final article](https://doi.org/10.1038/s41586-025-08704-w).
   [Final Supplementary Information](https://media.springernature.com/original/springer-static/esm/art%3A10.1038%2Fs41586-025-08704-w/MediaObjects/41586_2025_8704_MOESM1_ESM.pdf),
   especially §§4.4 and 4.6. The paper describes its cutoff model as a heuristic.
2. [Official version-2 data and analysis](https://doi.org/10.4121/6aa42f05-6823-4848-b235-3ea19e39f4ae.v2),
   including `dqc_simulation.ipynb` and `phys_epr_sim.py`. Repository license CC BY 4.0.
   The verified source archive SHA256 is `9c670972ffec0d49e0a41ac8c9fd5b755b9456f40d1a6a570db249be43b04e8f`.
3. The Qoala adapter is pinned to commit `31fa2c9699578d227736557f35cd01a134c4bbde`.
   API inspection and unchanged parent-module hashes are recorded in `source_provenance.json`.
